# 01 - Preprocessing

Run after `00_setup.ipynb`.

Reads the raw MATLAB output `image/*.png` + `label/*.txt` under `$DATA_ROOT/processed/{duke_dme,hc_ms}`
(written by `generate_dme_train.m` / `generate_hc_train.m`):

1. applies `src/s2_preprocessing/` (denoise, normalization; flattening/cropping already done) and
   writes `image/*.npy` + `label/*.txt` to `$DATA_ROOT/processed/{duke_dme,hc_ms}_denoised`
2. writes patient-level splits over the denoised datasets.

In [ ]:
import os, pathlib, sys

root = pathlib.Path.cwd()
while not (root / "requirements.txt").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.paths import resolve_roots
DATA_ROOT, OUTPUT_ROOT = resolve_roots()

print('Repo root  :', root)
print('DATA_ROOT  :', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

In [ ]:
from src.s1_data.dataset import OCTDataset
from src.s1_data.splits import build_patient_groups, patient_folds, save_folds
from src.s2_preprocessing.pipeline import preprocess_and_save

# 1. Denoise + normalize (parallel across CPU cores), save as .npy --------
preprocess_and_save(DATA_ROOT / 'processed/duke_dme', DATA_ROOT / 'processed/duke_dme_denoised')
preprocess_and_save(DATA_ROOT / 'processed/hc_ms', DATA_ROOT / 'processed/hc_ms_denoised')

# 2. Patient-level 5-fold CV split, stratified by group (DME/HC/MS) --------
duke_dme = OCTDataset(DATA_ROOT / 'processed/duke_dme_denoised')
hc_ms = OCTDataset(DATA_ROOT / 'processed/hc_ms_denoised')
patient_groups = build_patient_groups(duke_dme.patient_ids(), hc_ms.patient_ids())
folds = patient_folds(patient_groups)

folds_path = DATA_ROOT / 'processed/folds.json'
save_folds(folds, folds_path)
print(f"OK   {len(folds)} folds over {len(patient_groups)} patients "
      f"({len(duke_dme.patient_ids())} DME, {len(hc_ms.patient_ids())} HC-MS) -> {folds_path}")

# 02_run_methods.ipynb reads folds.json directly (src.s1_data.splits.load_folds)
# instead of recomputing — it doesn't depend on rerunning this notebook.